# Traitement automatisé — toutes les visites (V0, V1, V3, V5, Vc)



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce

## + UPPS 

In [ ]:
df_UPPS = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="UPPS")
# df_UPPS.head()
# Séparer le dataset par valeur de la colonne TITRE
df_v0_UPPS = df_UPPS[df_UPPS["TITRE"] == "Visite de screening"].reset_index(drop=True)
df_v1_UPPS = df_UPPS[df_UPPS["TITRE"] == "Visite Bilan à 1 an - V1"].reset_index(drop=True)
df_v3_UPPS = df_UPPS[df_UPPS["TITRE"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)
df_v5_UPPS = df_UPPS[df_UPPS["TITRE"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)


df_v0_UPPS.drop(columns=["TITRE","INIT_PAT"],axis=1,inplace=True)
df_v0_UPPS.insert(0, "VISITE", 0)
df_v1_UPPS.drop(columns=["TITRE","INIT_PAT"],axis=1,inplace=True)
df_v1_UPPS.insert(0, "VISITE", 1)
df_v3_UPPS.drop(columns=["TITRE","INIT_PAT"],axis=1,inplace=True)
df_v3_UPPS.insert(0, "VISITE", 3)
df_v5_UPPS.drop(columns=["TITRE","INIT_PAT"],axis=1,inplace=True)
df_v5_UPPS.insert(0, "VISITE", 5)



---
##  nettoyage des codes manquants (".D", ".A", ".K", ".F", "D...")

In [2]:
def nettoyer_codes_manquants(df, prefixes=("." ,)):
    """
    Repère toutes les valeurs texte commençant par l'un des préfixes
    dans les colonnes object du DataFrame, puis les remplace par NaN.
    Retourne le DataFrame nettoyé (inplace).
    """
    text_cols = df.select_dtypes(include="object").columns
    all_unique = set()
    for col in text_cols:
        all_unique.update(df[col].dropna().unique())

    codes = [v for v in all_unique
             if any(str(v).startswith(p) for p in prefixes)]
    if codes:
        print(f"  → codes remplacés par NaN : {codes}")
    return df.replace(codes, np.nan)



---
## Fonction principale : `traiter_visite(v, vc=False)`

In [15]:
def traiter_visite(v, vc=False):

    chemin = f"Output/version_2/V{v}.xlsx"
    print(f"\n{'='*60}")
    print(f"  TRAITEMENT  V{v}  —  {chemin}")
    print(f"{'='*60}")

    # ==================================================================
    # FEUILLES COMMUNES  (présentes dans toutes les visites, y compris Vc)
    # ==================================================================
     
    df_V = pd.read_excel(chemin, sheet_name=f"V{v}")
    df_V = nettoyer_codes_manquants(df_V)

    # ── LEDD ──────────────────────────────────────────────────────────
    df_LEDD = pd.read_excel(chemin, sheet_name="LEDD")
    df_LEDD["somme_calc"] = df_LEDD.iloc[:, 2:-1].sum(axis=1, skipna=True)
    df_LEDD["statut"] = np.where(
        np.isclose(df_LEDD["somme_calc"], df_LEDD["ledd_tot"], atol=0.01),
        "ok", "différent"
    )
    df_LEDD["ledd_tot"] = df_LEDD["somme_calc"]
    df_LEDD.drop(columns=["somme_calc", "statut"], inplace=True)
    df_Total_LEDD = df_LEDD[["SUBJID", "ledd_tot"]]
    print(f"df_LEDD                 : {df_LEDD.shape}")

    # ── PSYCHOTROPES ──────────────────────────────────────────────────
    df_PSYCHOTROPES = pd.read_excel(chemin, sheet_name="PSYCHOTROPES")
    print(f"\nFeuille PSYCHOTROPES    : {df_PSYCHOTROPES.shape}")
    df_PSYCHOTROPES = nettoyer_codes_manquants(df_PSYCHOTROPES)
    cols = df_PSYCHOTROPES.columns[:2].tolist()
    cols += [c for c in df_PSYCHOTROPES.columns if c.startswith('MEDICMT') or c.startswith('POSO')]
    df_PSYCHOTROPES = df_PSYCHOTROPES[cols]
    print(f"df_PSYCHOTROPES         : {df_PSYCHOTROPES.shape}")

    # ── AUTRE_PARKINSON ───────────────────────────────────────────────
    df_AUTRE_PARKINSON = pd.read_excel(chemin, sheet_name="AUTRE_PARKINSON")
    df_AUTRE_PARKINSON = nettoyer_codes_manquants(df_AUTRE_PARKINSON)
    cols = df_AUTRE_PARKINSON.columns[:2].tolist()
    cols += [c for c in df_AUTRE_PARKINSON.columns if c.startswith('MEDICMT') or c.startswith('POSO')]
    df_AUTRE_PARKINSON = df_AUTRE_PARKINSON[cols]
    print(f"df_AUTRE_PARKINSON      : {df_AUTRE_PARKINSON.shape}")

    # ── CONSO_SPECIFIQUE ──────────────────────────────────────────────
    df_CONSO_SPECIFIQUE = pd.read_excel(chemin, sheet_name="CONSO_SPECIFIQUE")
    df_CONSO_SPECIFIQUE = nettoyer_codes_manquants(df_CONSO_SPECIFIQUE)
    cols = df_CONSO_SPECIFIQUE.columns[:2].tolist()
    cols += [c for c in df_CONSO_SPECIFIQUE.columns if c.startswith('MEDICMT') or c.startswith('POSO')]
    df_CONSO_SPECIFIQUE = df_CONSO_SPECIFIQUE[cols]
    print(f"df_CONSO_SPECIFIQUE     : {df_CONSO_SPECIFIQUE.shape}")

    # ==================================================================
    # CAS SPÉCIAL  →  Vc  (uniquement les feuilles communes)
    # ==================================================================
    if vc:
        return {
            f"V{v}"             : df_V,
            "LEDD"             : df_LEDD,
            "CONSO_SPECIFIQUE" : df_CONSO_SPECIFIQUE,
            "PSYCHOTROPES"     : df_PSYCHOTROPES,
            "AUTRE_PARKINSON"  : df_AUTRE_PARKINSON,
        }

    # ==================================================================
    # FEUILLES SPÉCIFIQUES  V0 → V5
    # ==================================================================

    # ── UPDRSIII ──────────────────────────────────────────────────────
    if v in (0, 1):
        df_UPDRSIII = pd.read_excel(chemin, sheet_name="UPDRSIII")
        print(f"\nFeuille UPDRSIII        : {df_UPDRSIII.shape}")

        df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=(".",))
        df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=("D",))

        n_last = 7 if v == 0 else 4
        df_Total_UPDRSIII = df_UPDRSIII.iloc[:, [1] + list(range(-n_last, 0))]
        print(f"df_UPDRSIII             : {df_UPDRSIII.shape}")

        # ── UPDRSIII_S  (uniquement V0) ───────────────────────────────
        # Calcul du BEST ON, dopasensibilité, à partir de df_UPDRSIII
        if v == 0:
            TEMPS = [15, 30, 45, 60, 90, 120]

            # Mapping Convention B : suffixe index → temps en minutes
            SUFFIXE_B_TO_TEMPS = {"": 15, "1": 30, "2": 45, "3": 60, "4": 90, "5": 120}
            TEMPS_TO_SUFFIXE_B = {v_: k for k, v_ in SUFFIXE_B_TO_TEMPS.items()}

            df_UPDRSIII_S = df_UPDRSIII.copy()
            df_cols = list(df_UPDRSIII_S.columns)

            # Préfixes à exclure de la détection d'items
            EXCLUDED_PREFIXES = ("SS", "TOT", "SSTOT")

            # Détecter les items à partir des colonnes OFF_* (hors OFF_H et SS*/TOT*/SSTOT*)
            import re
            items = [
                re.sub(r'^OFF_', '', c, flags=re.IGNORECASE)
                for c in df_cols
                if re.match(r'^OFF_', c, re.IGNORECASE)
                and c.upper() != 'OFF_H'
                and not any(c.upper().startswith(p) for p in EXCLUDED_PREFIXES)
            ]

            def get_on_col(item, t):
                """
                Retourne le nom de colonne ON pour un item et un temps donnés.
                Teste d'abord la Convention A (ON_{item}15/30/.../120),
                puis la Convention B (ON_{item} / ON_{item}1 / ... / ON_{item}5).
                Retourne None si aucune colonne trouvée.
                """
                # Convention A : ex. ON_MIGCHE_PIED30
                col_a = f"ON_{item}{t}"
                if col_a in df_cols:
                    return col_a
                # Convention B : ex. ON_MIDROIT_JAMBE1 pour t=30
                suffix_b = TEMPS_TO_SUFFIXE_B.get(t, None)
                if suffix_b is not None:
                    col_b = f"ON_{item}{suffix_b}"
                    if col_b in df_cols:
                        return col_b
                return None

            # Convertir toutes les colonnes de scores en numérique
            off_cols = [f"OFF_{item}" for item in items if f"OFF_{item}" in df_cols]
            on_cols_all = [
                get_on_col(item, t)
                for item in items for t in TEMPS
                if get_on_col(item, t) is not None
            ]
            score_cols = off_cols + on_cols_all
            df_UPDRSIII_S[score_cols] = df_UPDRSIII_S[score_cols].apply(pd.to_numeric, errors="coerce")

            # 1. Score total OFF
            df_UPDRSIII_S["SCORE_TOTAL_OFF"] = df_UPDRSIII_S[off_cols].sum(axis=1, min_count=1)

            # 2. Scores totaux ON par temps
            for t in TEMPS:
                on_cols_t = [get_on_col(item, t) for item in items if get_on_col(item, t) is not None]
                if on_cols_t:
                    df_UPDRSIII_S[f"SCORE_TOTAL_ON_{t}"] = df_UPDRSIII_S[on_cols_t].sum(axis=1, min_count=1)

            # 3 & 4. BEST ON et temps correspondant
            on_total_cols = [f"SCORE_TOTAL_ON_{t}" for t in TEMPS if f"SCORE_TOTAL_ON_{t}" in df_UPDRSIII_S.columns]
            on_matrix = df_UPDRSIII_S[on_total_cols]
            df_UPDRSIII_S["BEST_ON"]      = on_matrix.min(axis=1)
            best_col                       = on_matrix.idxmin(axis=1)   # ex: "SCORE_TOTAL_ON_30"
            df_UPDRSIII_S["BEST_ON_TEMPS"] = best_col.str.extract(r'(\d+)$').astype(float)

            # 5. Conserver les items ON au temps BEST ON → colonnes BEST_ON_{item}
            all_on_item_cols = []
            for item in items:
                best_vals = []
                for _, row in df_UPDRSIII_S.iterrows():
                    best_t = row.get("BEST_ON_TEMPS")
                    if pd.isna(best_t):
                        best_vals.append(np.nan)
                        continue
                    col_name = get_on_col(item, int(best_t))
                    best_vals.append(row[col_name] if col_name and col_name in row.index else np.nan)
                df_UPDRSIII_S[f"BEST_ON_{item}"] = best_vals
                # Marquer toutes les colonnes ON_{item}{t} pour suppression
                for t in TEMPS:
                    c = get_on_col(item, t)
                    if c and c in df_UPDRSIII_S.columns:
                        all_on_item_cols.append(c)

            # Supprimer les colonnes ON par temps (maintenant redondantes)
            df_UPDRSIII_S.drop(columns=list(set(all_on_item_cols)), errors="ignore", inplace=True)

            # 6. Dopasensibilité
            off_num = pd.to_numeric(df_UPDRSIII_S["SCORE_TOTAL_OFF"], errors="coerce")
            bon_num = pd.to_numeric(df_UPDRSIII_S["BEST_ON"],         errors="coerce")
            df_UPDRSIII_S["DOPASENSIBILITE_PCT"] = np.where(
                off_num == 0,
                np.nan,
                (off_num - bon_num) / off_num * 100
            )

            # Supprimer les colonnes SS*, TOT*, SSTOT*
            cols_to_drop = [c for c in df_UPDRSIII_S.columns
                            if re.match(r'^(SS|TOT|SSTOT)', c, re.IGNORECASE)]
            df_UPDRSIII_S.drop(columns=cols_to_drop, errors="ignore", inplace=True)

            print(f"df_UPDRSIII_S           : {df_UPDRSIII_S.shape}")

    elif v in (3, 5):
        df_raw_v3v5 = pd.read_excel(
            "Data/Matthieu_Soumaya_Dec2025.xlsx",
            sheet_name="UPDRSIII_COMPLET_V3_V5 "
        )
        label_map = {
            3: "Visite Bilan à 3 ans - V3",
            5: "Visite Bilan à 5 ans - V5",
        }
        df_UPDRSIII = df_raw_v3v5[
            df_raw_v3v5["VISIT"] == label_map[v]
        ].reset_index(drop=True)

        df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=(".",))
        df_UPDRSIII = nettoyer_codes_manquants(df_UPDRSIII, prefixes=("D",))

        # Supprimer les 2 dernières colonnes redondantes
        # df_UPDRSIII.drop(columns=df_UPDRSIII.iloc[:, -2:].columns, inplace=True)
        df_UPDRSIII.drop("VISIT",axis=1,inplace=True)
        df_UPDRSIII.insert(0,"VISITE",v)

        cols_num = df_UPDRSIII.iloc[:, 4:].columns
        df_UPDRSIII[cols_num] = df_UPDRSIII[cols_num].apply(pd.to_numeric, errors="coerce")

        cols_score = df_UPDRSIII.iloc[:, 4:-2].columns
        cols_score = cols_score.drop(["SS_TOTAL1","ON_SS_TOT1"], errors="ignore")
        df_UPDRSIII["UPDRSIII_tot"] = df_UPDRSIII[cols_score].sum(axis=1, skipna=True)
        df_UPDRSIII["statut"] = np.where(
            np.isclose(df_UPDRSIII["UPDRSIII_tot"], df_UPDRSIII["ON_TOTAL"], atol=0.01),
            "ok", "différent"
        )
        df_Total_UPDRSIII = df_UPDRSIII[["SUBJID","UPDRSIII_tot"]]
        print(f"\ndf_UPDRSIII (V{v})       : {df_UPDRSIII.shape}")

    # ── IMC ───────────────────────────────────────────────────────────
    cols_v = df_V.iloc[:, -2:].columns
    df_V[cols_v] = df_V[cols_v].apply(pd.to_numeric, errors="coerce")
    df_V["TAILLE_m"] = np.where(df_V["TAILLE"] > 3, df_V["TAILLE"] / 100, df_V["TAILLE"])
    df_V["IMC"] = df_V["POIDS"] / (df_V["TAILLE_m"] ** 2)
    df_V.drop("TAILLE_m", axis=1, inplace=True)

    # ── UPDRSIV ───────────────────────────────────────────────────────
    df_UPDRSIV = pd.read_excel(chemin, sheet_name="UPDRSIV")
    print(f"\nFeuille UPDRSIV         : {df_UPDRSIV.shape}")
    df_UPDRSIV = nettoyer_codes_manquants(df_UPDRSIV)
    mapping_UPDRSIV = {"Normal": 0, "Minime": 1, "Léger": 2, "Modéré": 3, "Sévère": 4}
    df_UPDRSIV.iloc[:, 3:] = df_UPDRSIV.iloc[:, 3:].replace(mapping_UPDRSIV)
    df_UPDRSIV["UPDRSIV_tot"] = df_UPDRSIV.iloc[:, 2:].sum(axis=1, skipna=True)
    df_Total_UPDRSIV = df_UPDRSIV[["SUBJID", "UPDRSIV_tot"]]
    print(f"df_UPDRSIV              : {df_UPDRSIV.shape}")

    # ── PDQ39 ─────────────────────────────────────────────────────────
    df_PDQ39 = pd.read_excel(chemin, sheet_name="PDQ39")
    print(f"\nFeuille PDQ39           : {df_PDQ39.shape}")
    df_PDQ39 = nettoyer_codes_manquants(df_PDQ39)
    mapping_PDQ39 = {
        "Jamais": 0, "Rarement": 1, "Parfois": 2, "Souvent": 3,
        "Toujours": 4, "Toujours ou ne peut jamais faire": 4,
        "Oui": 1, "Non": 0
    }
    cols_obj = df_PDQ39.select_dtypes(include="object").columns[1:]
    df_PDQ39[cols_obj] = df_PDQ39[cols_obj].replace(mapping_PDQ39)
    domains = {
        "mobility":      ["PDQ39_1","PDQ39_2","PDQ39_3","PDQ39_4","PDQ39_5","PDQ39_6","PDQ39_7","PDQ39_8","PDQ39_9","PDQ39_10"],
        "adl":           ["PDQ39_11","PDQ39_12","PDQ39_13","PDQ39_14","PDQ39_15","PDQ39_16"],
        "emotional":     ["PDQ39_17","PDQ39_18","PDQ39_19","PDQ39_20","PDQ39_21","PDQ39_22"],
        "stigma":        ["PDQ39_23","PDQ39_24","PDQ39_25","PDQ39_26"],
        "social":        ["PDQ39_27","PDQ39_28","PDQ39_29"],
        "cognition":     ["PDQ39_30","PDQ39_31","PDQ39_32","PDQ39_33"],
        "communication": ["PDQ39_34","PDQ39_35","PDQ39_36"],
        "bodily":        ["PDQ39_37","PDQ39_38","PDQ39_39"]
    }
    for domain, cols in domains.items():
        df_PDQ39[domain + "_score"] = df_PDQ39[cols].sum(axis=1) / (len(cols) * 4) * 100
    domain_cols = [d + "_score" for d in domains.keys()]
    df_PDQ39["PDQ39_SI"] = df_PDQ39[domain_cols].mean(axis=1)
    df_Total_PDQ39 = df_PDQ39[["SUBJID", "PDQ39_SI"]]
    print(f"df_PDQ39                : {df_PDQ39.shape}")

    # ── QUIP ──────────────────────────────────────────────────────────
    df_QUIP = pd.read_excel(chemin, sheet_name="QUIP")
    print(f"\nFeuille QUIP            : {df_QUIP.shape}")
    df_QUIP = nettoyer_codes_manquants(df_QUIP)

    # ── MOCA ──────────────────────────────────────────────────────────
    df_MOCA = pd.read_excel(chemin, sheet_name="MOCA")
    print(f"\nFeuille MOCA            : {df_MOCA.shape}")
    df_MOCA = nettoyer_codes_manquants(df_MOCA)
    cols_moca = df_MOCA.iloc[:, 2:].columns
    df_MOCA[cols_moca] = df_MOCA[cols_moca].apply(pd.to_numeric, errors="coerce")
    df_MOCA["MOCA_tot"] = df_MOCA.iloc[:, 2:-1].sum(axis=1, skipna=True)
    df_MOCA["statut"] = np.where(
        np.isclose(df_MOCA["MOCA_tot"], df_MOCA["MOCA_SCORE"], atol=0.01),
        "ok", "différent"
    )
    df_Total_MOCA = df_MOCA[["SUBJID","MOCA_tot"]]
    print(f"df_MOCA                 : {df_MOCA.shape}")

    # ── HAMA ──────────────────────────────────────────────────────────
    df_HAMA = pd.read_excel(chemin, sheet_name="HAMA")
    print(f"\nFeuille HAMA            : {df_HAMA.shape}")
    df_HAMA = nettoyer_codes_manquants(df_HAMA)
    mapping_HAMA = {
        "Absent": 0,
        "Léger": 1, "Anxiété légère": 1,
        "Modéré": 2, "Anxiété légère à modérée": 2,
        "Sévère": 3, "Anxiété modérée à grave": 3,
        "Très sévère": 4
    }
    cols_obj = df_HAMA.select_dtypes(include="object").columns[1:]
    df_HAMA[cols_obj] = df_HAMA[cols_obj].replace(mapping_HAMA)
    cols_hama = df_HAMA.iloc[:, 2:-2].columns
    df_HAMA[cols_hama] = df_HAMA[cols_hama].apply(pd.to_numeric, errors="coerce")
    df_HAMA["HAMA_SCORECALC"] = df_HAMA["HAMA_SCORECALC"].apply(pd.to_numeric, errors="coerce")
    df_HAMA["HAMA_tot"] = df_HAMA[cols_hama].sum(axis=1, skipna=True)
    df_HAMA["statut"] = np.where(
        np.isclose(df_HAMA["HAMA_tot"], df_HAMA["HAMA_SCORECALC"], atol=0.01),
        "ok", "différent"
    )
    col = df_HAMA.pop("ANXIETE")
    df_HAMA["ANXIETE"] = col
    df_Total_HAMA = df_HAMA[["SUBJID","HAMA_tot"]]
    print(f"df_HAMA                 : {df_HAMA.shape}")

    # ── HAMD ──────────────────────────────────────────────────────────
    df_HAMD = pd.read_excel(chemin, sheet_name="HAMD")
    print(f"\nFeuille HAMD            : {df_HAMD.shape}")
    df_HAMD = nettoyer_codes_manquants(df_HAMD)
    cols_hamd = df_HAMD.iloc[:, 2:-2].columns
    df_HAMD["HAMD_tot"] = df_HAMD[cols_hamd].sum(axis=1, skipna=True)
    df_HAMD["statut"] = np.where(
        np.isclose(df_HAMD["HAMD_tot"], df_HAMD["HAMD_SCORECALC"], atol=0.01),
        "ok", "différent"
    )
    col = df_HAMD.pop("DEPRESSION")
    df_HAMD["DEPRESSION"] = col
    mapping_HAMD = {
        "Symptomes dépressifs légers": 1,
        "Symptômes dépressifs légers à modérés": 2,
        "Symptômes dépressifs modérés à sévères": 3,
    }
    cols_obj = df_HAMD.select_dtypes(include="object").columns[-1:]
    df_HAMD[cols_obj] = df_HAMD[cols_obj].replace(mapping_HAMD)
    df_Total_HAMD = df_HAMD[["SUBJID", "HAMD_tot"]]
    print(f"df_HAMD                 : {df_HAMD.shape}")

    # ── LARS ──────────────────────────────────────────────────────────
    df_LARS = pd.read_excel(chemin, sheet_name="LARS")
    print(f"\nFeuille LARS            : {df_LARS.shape}")
    df_LARS = nettoyer_codes_manquants(df_LARS)
    cols_lars = df_LARS.iloc[:, 2:-2].columns
    df_LARS["LARS_tot"] = df_LARS[cols_lars].sum(axis=1, skipna=True)
    df_LARS["statut"] = np.where(
        np.isclose(df_LARS["LARS_tot"], df_LARS["LARS_SCORE"], atol=0.01),
        "ok", "différent"
    )
    col = df_LARS.pop("LARS_RESULTAT")
    df_LARS["LARS_RESULTAT"] = col
    mapping_LARS = {
        "Non apathique": 1,
        "Tendance à l'apathie": 2,
        "Apathie modérée": 3,
        "Apathie sévère": 4,
    }
    cols_obj = df_LARS.select_dtypes(include="object").columns[-1:]
    df_LARS[cols_obj] = df_LARS[cols_obj].replace(mapping_LARS)
    df_Total_LARS = df_LARS[["SUBJID", "LARS_tot"]]
    print(f"df_LARS                 : {df_LARS.shape}")

    # ── ECMP ──────────────────────────────────────────────────────────
    df_ECMP = pd.read_excel(chemin, sheet_name="ECMP")
    print(f"\nFeuille ECMP            : {df_ECMP.shape}")
    df_ECMP = nettoyer_codes_manquants(df_ECMP)

    # ── DIGITSMT_TRAILMT_DKEFS ────────────────────────────────────────
    df_DIGITSMT = pd.read_excel(chemin, sheet_name="DIGITSMT_TRAILMT_DKEFS")
    print(f"\nFeuille DIGITSMT        : {df_DIGITSMT.shape}")
    df_DIGITSMT = nettoyer_codes_manquants(df_DIGITSMT)

    # ── UPPS (fichier externe, V0 V1 V3 V5) ──────────────────────────
    if v in (0, 1, 3, 5):
        label_map_upps = {
            0: "Visite de screening",
            1: "Visite Bilan à 1 an - V1",
            3: "Visite Bilan à 3 ans - V3",
            5: "Visite Bilan à 5 ans - V5",
        }
        df_raw_upps = pd.read_excel(
            "Data/Matthieu_Soumaya_Dec2025.xlsx",
            sheet_name="UPPS"
        )
        df_UPPS = df_raw_upps[
            df_raw_upps["TITRE"] == label_map_upps[v]
        ].reset_index(drop=True)
        df_UPPS.drop(columns=["TITRE", "INIT_PAT"], inplace=True)
        df_UPPS.insert(0, "VISITE", v)
        df_UPPS = nettoyer_codes_manquants(df_UPPS)
        print(f"df_UPPS                 : {df_UPPS.shape}")

    # ── FREQUENCE  (V1, V2, V3) ───────────────────────────────────────
    if v in (1, 2, 3):
        df_FREQUENCE = pd.read_excel(chemin, sheet_name="FREQUENCE")
        df_FREQUENCE = nettoyer_codes_manquants(df_FREQUENCE)

    # ==================================================================
    # TABLE DES TOTAUX
    # ==================================================================
    lis_totaux = [
        df_Total_LEDD,
        df_Total_UPDRSIII,
        df_Total_UPDRSIV,
        df_Total_MOCA,
        df_Total_PDQ39,
        df_Total_HAMA,
        df_Total_HAMD,
        df_Total_LARS,
    ]
    lis_avec_id = [d for d in lis_totaux if "SUBJID" in d.columns]
    if lis_avec_id:
        df_Totaux = reduce(
            lambda left, right: pd.merge(left, right, on="SUBJID", how="outer"),
            lis_avec_id
        )
        df_Totaux = pd.concat(
            [df_Totaux.iloc[[-1]], df_Totaux.iloc[:-1]], ignore_index=True
        )
    else:
        df_Totaux = pd.DataFrame()

    # ==================================================================
    # CONSTRUCTION DU DICTIONNAIRE DE RETOUR
    # ==================================================================
    result = {
        f"V{v}"                   : df_V,
        "LEDD"                    : df_LEDD,
        "CONSO_SPECIFIQUE"        : df_CONSO_SPECIFIQUE,
        "PSYCHOTROPES"            : df_PSYCHOTROPES,
        "AUTRE_PARKINSON"         : df_AUTRE_PARKINSON,
        "UPDRSIII"                : df_UPDRSIII,
        "UPDRSIV"                 : df_UPDRSIV,
        "PDQ39"                   : df_PDQ39,
        "QUIP"                    : df_QUIP,
        "MOCA"                    : df_MOCA,
        "HAMA"                    : df_HAMA,
        "HAMD"                    : df_HAMD,
        "LARS"                    : df_LARS,
        "ECMP"                    : df_ECMP,
        "DIGITSMT_TRAILMT_DKEFS"  : df_DIGITSMT,
        "Totaux"                  : df_Totaux,
    }

    # ── UPDRSIII_S  →  ajoutée juste après UPDRSIII, uniquement pour V0 ──
    if v == 0:
        result["UPDRSIII_S"] = df_UPDRSIII_S

    if v in (0, 1, 3, 5):
        result["UPPS"] = df_UPPS

    if v in (1, 2, 3):
        result["FREQUENCE"] = df_FREQUENCE

    return result

In [4]:
ORDRE_FEUILLES = [
    
    "PDQ39",
    "UPDRSIII",
    "UPDRSIII_S",
    "UPDRSIV",
    
    "ECMP",
    "QUIP",
    "LARS",
    "UPPS",
    "HAMD",
    "HAMA",
    "MOCA",
    "DIGITSMT_TRAILMT_DKEFS",

    "FREQUENCE",
    
    "CONSO_SPECIFIQUE",
    "PSYCHOTROPES",
    "AUTRE_PARKINSON",

    "LEDD",
    "Totaux",
]

def reordonner_feuilles(sheets, v):
    cle_visite    = f"V{v}"
    ordre_complet = [cle_visite] + ORDRE_FEUILLES
    return {k: sheets[k] for k in ordre_complet if k in sheets}

---
## Utilitaire d'écriture Excel

In [5]:
def ecrire_excel(sheets_dict, version, output_dir="Output/version_3"):
    filepath = f"{output_dir}/V{version}.xlsx"
    with pd.ExcelWriter(filepath, engine="openpyxl") as writer:
        for sheet_name, df in sheets_dict.items():
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"[OK] {filepath}  →  {len(sheets_dict)} feuilles")

---
## Traitement + écriture de chaque visite

### Vc

In [16]:
sheets_Vc = traiter_visite("c", vc=True)
sheets_Vc = reordonner_feuilles(sheets_Vc, "c")
ecrire_excel(sheets_Vc, "c")


  TRAITEMENT  Vc  —  Output/version_2/Vc.xlsx
df_LEDD                 : (491, 7)

Feuille PSYCHOTROPES    : (835, 80)
  → codes remplacés par NaN : ['.D', '.A']
df_PSYCHOTROPES         : (835, 28)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → codes remplacés par NaN : ['.D', '.A', '.K', '.C']
df_AUTRE_PARKINSON      : (835, 28)
  → codes remplacés par NaN : ['.A']
df_CONSO_SPECIFIQUE     : (835, 28)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


[OK] Output/version_3/Vc.xlsx  →  5 feuilles


### V0

In [17]:
sheets_V0 = traiter_visite(0)
sheets_V0 = reordonner_feuilles(sheets_V0, 0)

ecrire_excel(sheets_V0, 0)



  TRAITEMENT  V0  —  Output/version_2/V0.xlsx
  → codes remplacés par NaN : ['.D']
df_LEDD                 : (794, 7)

Feuille PSYCHOTROPES    : (835, 80)
  → codes remplacés par NaN : ['.K', '.D', '.A']
df_PSYCHOTROPES         : (835, 28)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → codes remplacés par NaN : ['.K', '.D', '.C', '.A']
df_AUTRE_PARKINSON      : (835, 28)
  → codes remplacés par NaN : ['.K', '.D', '.A']
df_CONSO_SPECIFIQUE     : (835, 28)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)



Feuille UPDRSIII        : (836, 265)
  → codes remplacés par NaN : ['.K', '.D', '.F', '.A']
  → codes remplacés par NaN : ['DM:DM']
df_UPDRSIII             : (836, 265)


...


Feuille HAMD            : (835, 22)
df_HAMD                 : (835, 24)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\2257146840.py:329: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_HAMD[cols_obj] = df_HAMD[cols_obj].replace(mapping_HAMD)



Feuille LARS            : (835, 13)
df_LARS                 : (835, 15)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\2257146840.py:352: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_LARS[cols_obj] = df_LARS[cols_obj].replace(mapping_LARS)



Feuille ECMP            : (835, 44)
  → codes remplacés par NaN : ['.D', '.A']

Feuille DIGITSMT        : (835, 21)
  → codes remplacés par NaN : ['.F', '.D', '.A', '.K', '.C']
df_UPPS                 : (835, 22)
[OK] Output/version_3/V0.xlsx  →  18 feuilles


### V1

In [8]:
sheets_V1 = traiter_visite(1)
sheets_V1 = reordonner_feuilles(sheets_V1, 1)
ecrire_excel(sheets_V1, 1)


  TRAITEMENT  V1  —  Output/version_2/V1.xlsx
  → codes remplacés par NaN : ['.D', '.A']
df_LEDD                 : (525, 7)

Feuille PSYCHOTROPES    : (835, 80)
  → codes remplacés par NaN : ['.D', '.A']
df_PSYCHOTROPES         : (835, 28)
  → codes remplacés par NaN : ['.A', '.K']
df_AUTRE_PARKINSON      : (835, 28)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → codes remplacés par NaN : ['.D', '.A']
df_CONSO_SPECIFIQUE     : (835, 28)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)



Feuille UPDRSIII        : (836, 155)
  → codes remplacés par NaN : ['.K', '.D', '.F', '.A']
  → codes remplacés par NaN : ['Dose de L Dopa', 'DM:DM', 'DM']
df_UPDRSIII             : (836, 155)

Feuille UPDRSIV         : (835, 9)
  → codes remplacés par NaN : ['.D']
df_UPDRSIV              : (835, 10)


...


Feuille LARS            : (835, 13)
df_LARS                 : (835, 15)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\2884132872.py:319: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_LARS[cols_obj] = df_LARS[cols_obj].replace(mapping_LARS)



Feuille ECMP            : (835, 44)
  → codes remplacés par NaN : ['.D', '.A']

Feuille DIGITSMT        : (835, 21)
  → codes remplacés par NaN : ['.F', '.D', '.A', '.K']
df_UPPS                 : (835, 22)
  → codes remplacés par NaN : ['.6', '.K', '.D', '.C', '.F', '.A']


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


[OK] Output/version_3/V1.xlsx  →  18 feuilles


### V3

In [11]:
sheets_V3 = traiter_visite(3)
sheets_V3 = reordonner_feuilles(sheets_V3, 3)
ecrire_excel(sheets_V3, 3)


  TRAITEMENT  V3  —  Output/version_2/V3.xlsx
  → codes remplacés par NaN : ['.D', '.A']
df_LEDD                 : (272, 7)

Feuille PSYCHOTROPES    : (835, 80)
  → codes remplacés par NaN : ['.D', '.A']
df_PSYCHOTROPES         : (835, 28)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → codes remplacés par NaN : ['.D', '.A']
df_AUTRE_PARKINSON      : (835, 28)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → codes remplacés par NaN : ['.K', '.D', '.A']
df_CONSO_SPECIFIQUE     : (835, 28)


...


Feuille QUIP            : (835, 33)

Feuille MOCA            : (835, 14)
df_MOCA                 : (835, 16)

Feuille HAMA            : (835, 18)
df_HAMA                 : (835, 20)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\1936885456.py:264: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_HAMA[cols_obj] = df_HAMA[cols_obj].replace(mapping_HAMA)



Feuille HAMD            : (835, 22)
df_HAMD                 : (835, 24)

Feuille LARS            : (835, 13)
df_LARS                 : (835, 15)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\1936885456.py:296: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_HAMD[cols_obj] = df_HAMD[cols_obj].replace(mapping_HAMD)
C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\1936885456.py:319: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_LARS[cols_obj] = df_LARS[cols_obj].replace(mapping_LARS)



Feuille ECMP            : (835, 44)
  → codes remplacés par NaN : ['.D']

Feuille DIGITSMT        : (835, 21)
  → codes remplacés par NaN : ['.F', '.D', '.A', '.K']
df_UPPS                 : (835, 22)
  → codes remplacés par NaN : ['.K', '.D', '.F', '.A']
[OK] Output/version_3/V3.xlsx  →  18 feuilles


### V5

In [12]:
sheets_V5 = traiter_visite(5)
sheets_V5 = reordonner_feuilles(sheets_V5, 5)
ecrire_excel(sheets_V5, 5)


  TRAITEMENT  V5  —  Output/version_2/V5.xlsx
  → codes remplacés par NaN : ['.D', '.A']
df_LEDD                 : (313, 7)

Feuille PSYCHOTROPES    : (835, 80)
  → codes remplacés par NaN : ['.D', '.A', '.K']
df_PSYCHOTROPES         : (835, 28)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → codes remplacés par NaN : ['.D', '.A', '.K']
df_AUTRE_PARKINSON      : (835, 28)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\3651825493.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(codes, np.nan)


  → codes remplacés par NaN : ['.K', '.D', '.A']
df_CONSO_SPECIFIQUE     : (835, 28)


...


Feuille HAMD            : (835, 22)
df_HAMD                 : (835, 24)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\1936885456.py:296: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_HAMD[cols_obj] = df_HAMD[cols_obj].replace(mapping_HAMD)



Feuille LARS            : (835, 13)
df_LARS                 : (835, 15)


C:\Users\toufi\AppData\Local\Temp\ipykernel_8808\1936885456.py:319: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_LARS[cols_obj] = df_LARS[cols_obj].replace(mapping_LARS)



Feuille ECMP            : (835, 44)
  → codes remplacés par NaN : ['.D']

Feuille DIGITSMT        : (835, 21)
  → codes remplacés par NaN : ['.F', '.D', '.A', '.K']
df_UPPS                 : (835, 22)
[OK] Output/version_3/V5.xlsx  →  17 feuilles


# Prétraitement info statiques 

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
info =  pd.read_excel("Output/version_2/info.xlsx")
info.head()


,SUBJID,INIT_PAT,D_SCREEN,D_1ER_SYMPT,D_DIAG,D_LDOPA,D_TTT_DOPAM,D_FLUCTU_MOTR,D_FLUCTU_NONMOTR,D_DYSKINESIE,...,AGE,SEXE,SUIVI_ETUDE,D_FIN_ETUDE,D_SORTIE_PREMA,MOTIF_SORTIE_PREMA,AUTRE_PRECIS,D_INVESTIGATEUR,NOM_INVESTIGATEUR,MOTIF_SORTIE_PREMA1
0,Subject Identifier for the Study,initiales patient,date de la visite de screening,année premiers symptomes,année du diagnostic de la maladie,année d'introduction de la L-DOPA,année d'introduction du traitement dopaminergique,année d'apparition des fluctuations motrices,année d'apparition des fluctuations non motrices,année d'apparition des dyskinésies,...,âge,sexe,suivi étude,date de fin de l'étude,date de sortie prématurée,motif de sortie prématurée,autre précision,date de signature de l'investigateur,nom de l'investigateur,motif de sortie prématurée LIB
1,01-001,SR,18/11/2013,1998,1999,2000,2000,2002,2000,.D,...,67,1,0,NaN,22/11/2013,1,NaN,18/03/2014,MOREAU CAROLINE,Patient non opéré
2,01-002,TM,13/01/2014,2006,2006,2006,2006,2011,.K,.K,...,59,1,0,NaN,20/02/2014,6,SUSPICION DE CANCER PULMONAIRE,24/03/2014,DR MOREAU,Autre
3,01-003,SJ,04/03/2014,2001,2001,2003,2001,2004,.K,2004,...,61,1,0,NaN,07/03/2014,1,NaN,21/03/2014,DR HOPES LUCIE,Patient non opéré
4,01-004,DJ,12/05/2014,1998,2000,2003,2000,2003,2003,2003,...,65,2,0,NaN,09/01/2015,6,RETRAIT DU MATERIEL ET RETRAIT DE CONSENTEMENT,09/01/2015,DEVOS,Autre


In [12]:
info.drop("D_SCREEN",axis=1,inplace=True)
info = nettoyer_codes_manquants(info)
info.to_excel("Output/version_3/info.xlsx",index=False)


  → codes remplacés par NaN : ['.K', '.D', '.A', '.C']
